# Offer Comparator

Use of a sandbox to read and write files.
Define tools to get the content of the offers as .txt files in the sandbox.
Use of structured output.
Simple agent example to analyze one offer.
Deep agent example using built in todo list, Tavily search tool and a skill


In [ ]:
import os, sys

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool
from langchain_tavily import TavilySearch
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from pydantic import BaseModel, Field
from langgraph.checkpoint.memory import MemorySaver

In [ ]:
load_dotenv(override=True)

sandbox = os.path.abspath("sandbox")
os.makedirs(sandbox, exist_ok=True)

model = ChatOpenAI(model="gpt-5.4-mini")

# Initialize the Tavily Search tool
tavily_search = TavilySearch(max_results=5, topic="general")

## Tools to get the offer list and the content

In [ ]:
def read_offer_file(filename):
    with open(filename, 'r') as file:
        return file.read()

@tool
def read_offer_file_tool(filename: str) -> str:
    """Reads the content of a file.
    Args:
        filename (str): The name of the file to read.
    Returns:
        str: The content of the file.
    """
    return read_offer_file("./sandbox/" + filename)

@tool
def get_offer_names() -> list:
    """Returns a list of offer file names."""
    offer_names = [
        filename for filename in os.listdir(sandbox)
        if filename.endswith(".txt") and os.path.isfile(os.path.join(sandbox, filename))
    ]
    return offer_names

## Structured output for agents

In [ ]:
class Offers(BaseModel):
    names: list[str] = Field(description="List of offer file names.")

class Offer(BaseModel):
    offer_name: str = Field(description="Offer file name.")
    concerns_PV: bool = Field(description="Whether the offer is related to photovoltaics (PV) and wallbox")
    concerns_WP: bool = Field(description="Whether the offer is related to heat pumps (WP)")
    PV_offer: str = Field(default="Not able to provide details", 
                          description="The photovoltaic (PV) and wallbox related offer details formatted as markdown, starting with the supplier name")
    WP_offer: str = Field(default="Not able to provide details", 
                          description="The heat pump (WP) related offer details formatted as markdown, starting with the supplier name")


## We first use classic agents

In [ ]:
agent = create_agent(
    model="openai:gpt-5.4-mini",
    tools=[read_offer_file_tool, get_offer_names],
    system_prompt="You are a helpful assistant who answers concisely.",
    response_format=Offers,
    checkpointer=MemorySaver(),
)

config = {"configurable": {"thread_id": "classic-agent"}}
result = agent.invoke({"messages": [{"role": "user", "content": "Give me the list of offers."}]}, config=config)
print(result["messages"][-1].content)

In [ ]:
agent = create_agent(
    model="openai:gpt-5.4-mini",
    tools=[read_offer_file_tool, get_offer_names],
    system_prompt="You are a helpful assistant who answers concisely.",
    response_format=Offer,
)
config = {"configurable": {"thread_id": "classic-agent"}}
result = agent.invoke({"messages": [{"role": "user", "content": "Give me the content of enpal.txt."}]}, config=config)
print(result["messages"][-1].content)
report = result["structured_response"]
print(report.offer_name)
print(report.concerns_PV)
print(report.concerns_WP)
print(report.PV_offer)
print(report.WP_offer)

Get the list of tools used by the agent

In [ ]:
tools_used = [tc["name"] for m in result["messages"] for tc in (getattr(m, "tool_calls", []) or [])]
print("Tools the lead agent called:", tools_used)

## The deep_agent has a builtin todo tool and sub-agents which will be used

Also contains tools to access files. I had to mention that the agent should avoid using the built-in tools like `ls` or `read-file`

In [ ]:
subagent_instructions = """
You research one offer with the search tool to check if the offer meets the state of the art. 
You also check the prices of the offer and compare them to the market prices. 
Use your skills to get the workflow.
"""

deep_agent_instructions = """
You are a research analyst. Plan your work with your todo tool.
Get the offer names from your tool `get_offer_names`. Avoid using built-in tools like ls or read-file.
For each offer name, delegate the offer reading and research about the market state of the art and prices to your sub-agent.
As you get all informations from your delegated agent, write your comparison report as a tidy markdown briefing to a file and store it in the sandbox folder.
"""

research_subagent = {
    "name": "Offer-researcher",
    "description": "Researches a single offer for state of the art and price. Use provided tools to read the offer and research it. Avoid using built-in tools like ls or read-file.",
    "system_prompt": subagent_instructions,
    "tools": [tavily_search, read_offer_file_tool],
    "skills": ["/skills/"],
}

researcher = create_deep_agent(
    model=model,
    tools=[tavily_search, get_offer_names],
    system_prompt=(deep_agent_instructions),
    subagents=[research_subagent],
    backend=FilesystemBackend(root_dir=sandbox, virtual_mode=True),
)

In [ ]:
mission = """
Compare the available offers, then write a detailed markdown report about the different solutions.
"""

config = {"configurable": {"thread_id": "compare-offers"}}

result = researcher.invoke({"messages": [{"role": "user", "content": mission}]},  config=config)
print(result["messages"][-1].content)

## Analyze the output and langgraph graph

In [ ]:
tools_used = [tc["name"] for m in result["messages"] for tc in (getattr(m, "tool_calls", []) or [])]
print("Tools the lead agent called:", tools_used)

In [ ]:
from langchain_core.runnables.graph_mermaid import draw_mermaid_png
from IPython.display import Image
mm = researcher.get_graph().draw_mermaid().replace("[", " ").replace("]", " ")
Image(draw_mermaid_png(mm, output_file_path="graph.png"))